In [1]:
!pip install requests

No global/local python version has been set yet. Please set the global/local version by typing:
pyenv global 3.7.4
pyenv local 3.7.4


### Les coordonnées, URL et périodes

In [2]:
import requests

coordonnees = {
"Albigny-sur-Saône": (45.874994, 4.833002),
    "Bron": (45.733925, 4.912376),
    "Caluire-et-Cuire": (45.788408, 4.840143),
    "Champagne-au-Mont-d'Or": (45.794851, 4.792018),
    "Chassieu": (45.738192, 4.969525),
    "Collonges-au-Mont-d'Or": (45.826654, 4.843424),
    "Couzon-au-Mont-d'Or": (45.845998, 4.832412),
    "Décines-Charpieu": (45.771873, 4.955767),
    "Fontaines-sur-Saône": (45.833936, 4.847240),
    "La Mulatière": (45.727369, 4.814635),

    "Lyon 1er Arrondissement": (45.768952, 4.830784),
    "Lyon 2e Arrondissement": (45.752383, 4.828261),
    "Lyon 3e Arrondissement": (45.754844, 4.863078),
    "Lyon 4e Arrondissement": (45.777987, 4.825779),
    "Lyon 5e Arrondissement": (45.758478, 4.809764),
    "Lyon 6e Arrondissement": (45.770945, 4.851613),
    "Lyon 7e Arrondissement": (45.741779, 4.840139),
    "Lyon 8e Arrondissement": (45.737008, 4.869250),
    "Lyon 9e Arrondissement": (45.779867, 4.807047),

    "Neuville-sur-Saône": (45.873743, 4.839111),
    "Oullins": (45.717045, 4.806313),
    "Pierre-Bénite": (45.703256, 4.821200),
    "Rillieux-la-Pape": (45.810883, 4.886695),
    "Saint-Cyr-au-Mont-d'Or": (45.799791, 4.819602),
    "Saint-Didier-au-Mont-d'Or": (45.791574, 4.808535),
    "Saint-Fons": (45.709265, 4.857221),
    "Saint-Genis-Laval": (45.699345, 4.799343),
    "Saint-Germain-au-Mont-d'Or": (45.887543, 4.803977),
    "Saint-Priest": (45.710196, 4.916000),
    "Sainte-Foy-lès-Lyon": (45.749565, 4.800896),
    "Tassin-la-Demi-Lune": (45.763256, 4.778610),
    "Vaulx-en-Velin": (45.771718, 4.921055),
    "Villeurbanne": (45.769355, 4.884227),
    "Vénissieux": (45.709458, 4.872295),
    "Écully": (45.775089, 4.778673),
}

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

periodes = [
    ("2023-01-01", "2023-12-31"),
    ("2024-01-01", "2024-12-31"),
    ("2025-01-01", "2025-12-31"),
    ("2026-01-01", "2026-08-29"),
]

### variables météo

In [3]:
variables_meteo = (
    "temperature_2m,"
    "relative_humidity_2m,"
    "apparent_temperature,"
    "precipitation,"
    "rain,"
    "snowfall,"
    "weather_code,"
    "wind_speed_10m,"
    "wind_gusts_10m,"
    "is_day,"
    "visibility"
)

### fonction qui appelle Open-Meteo

In [4]:
def recuperer_meteo(latitude, longitude, start_date, end_date):
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": start_date,
        "end_date": end_date,
        "minutely_15": variables_meteo,
    }

    response = requests.get(
        url,
        params=params,
        timeout=180,
    )

    response.raise_for_status()

    return response.json()

### créer le dossier où stocker les JSON

In [5]:
from pathlib import Path

dossier_json = Path("data/weather_raw")
dossier_json.mkdir(parents=True, exist_ok=True)

print(dossier_json)

data\weather_raw


### fonction pour créer un nom de fichier propre

In [6]:
import re
import unicodedata


def normaliser_nom(nom):
    nom = unicodedata.normalize("NFKD", nom)
    nom = nom.encode("ascii", "ignore").decode("ascii")
    nom = nom.lower()
    nom = re.sub(r"[^a-z0-9]+", "_", nom)

    return nom.strip("_")

In [7]:
# tester:

print(normaliser_nom("Décines-Charpieu"))
print(normaliser_nom("Lyon 1er Arrondissement"))

decines_charpieu
lyon_1er_arrondissement


### boucle complète

In [8]:
import json
import time

for commune, (latitude, longitude) in coordonnees.items():

    for start_date, end_date in periodes:

        print(
            f"Récupération : {commune} "
            f"| {start_date} → {end_date}"
        )

        try:
            data = recuperer_meteo(
                latitude,
                longitude,
                start_date,
                end_date,
            )

            document = {
                "commune": commune,
                "requested_latitude": latitude,
                "requested_longitude": longitude,
                "start_date": start_date,
                "end_date": end_date,
                "source": "open-meteo",
                "granularity": "15_min",
                "data": data,
            }

            nom_commune = normaliser_nom(commune)
            annee = start_date[:4]

            fichier = dossier_json / (
                f"{nom_commune}_{annee}.json"
            )

            with fichier.open(
                "w",
                encoding="utf-8",
            ) as f:
                json.dump(
                    document,
                    f,
                    ensure_ascii=False,
                )

            nombre = len(
                data["minutely_15"]["time"]
            )

            print(
                f"✅ {nombre} observations "
                f"→ {fichier}"
            )

        except requests.RequestException as error:
            print(
                f"❌ Erreur pour {commune} "
                f"{start_date} : {error}"
            )

        time.sleep(1)

Récupération : Albigny-sur-Saône | 2023-01-01 → 2023-12-31
❌ Erreur pour Albigny-sur-Saône 2023-01-01 : 429 Client Error: Too Many Requests for url: https://historical-forecast-api.open-meteo.com/v1/forecast?latitude=45.874994&longitude=4.833002&start_date=2023-01-01&end_date=2023-12-31&minutely_15=temperature_2m%2Crelative_humidity_2m%2Capparent_temperature%2Cprecipitation%2Crain%2Csnowfall%2Cweather_code%2Cwind_speed_10m%2Cwind_gusts_10m%2Cis_day%2Cvisibility
Récupération : Albigny-sur-Saône | 2024-01-01 → 2024-12-31
❌ Erreur pour Albigny-sur-Saône 2024-01-01 : 429 Client Error: Too Many Requests for url: https://historical-forecast-api.open-meteo.com/v1/forecast?latitude=45.874994&longitude=4.833002&start_date=2024-01-01&end_date=2024-12-31&minutely_15=temperature_2m%2Crelative_humidity_2m%2Capparent_temperature%2Cprecipitation%2Crain%2Csnowfall%2Cweather_code%2Cwind_speed_10m%2Cwind_gusts_10m%2Cis_day%2Cvisibility
Récupération : Albigny-sur-Saône | 2025-01-01 → 2025-12-31
❌ Erreur 